# 01. Decorators

**Tier:** Deep. **Why you'd use this:** You want the same timing or logging wrapper on several functions without pasting the wrapper by hand. `@decorator` is the short spelling of `f = decorator(f)`.

**After this lesson you can:** say what a wrapper returns; write `@decorator` as `f = decorator(f)`; keep the original name with `functools.wraps`; build a decorator that takes arguments.
About 30 minutes. **Uses:** [Scope](../05-functions/03-scope-lesson.ipynb), [Dataclasses](../11-classes/02-dataclasses-lesson.ipynb)


## Quick review

1. What does a nested function close over?
2. Why might you care what `__name__` a wrapped function still has?

<details><summary>Answers</summary>

Names from the enclosing scope that it reads later. Logs, docs, and debuggers use the original name — `wraps` keeps it.
</details>


## The idea

A decorator is a function that takes a function and returns a new one. The `@deco` line above a `def` is sugar for `name = deco(name)` after the definition. The returned function usually closes over the original and runs extra work before or after calling it. `functools.wraps` copies the original's name and docstring onto the wrapper so tools and errors still show the real function.

<details><summary>Explain more</summary>

Think of a gift wrap: the present is still inside, but callers see the wrapping first. `@shout` above `def plain` means `plain = shout(plain)`. A decorator with arguments is one more layer: `@twice(3)` first calls `twice(3)` to build the real decorator, then that decorator wraps the function.
</details>


Stacking decorators applies bottom-up: the decorator closest to `def` wraps first, then the one above wraps that result. Reading `@a` / `@b` / `def f` is `f = a(b(f))`. Order matters when both wrappers care about arguments or return values.


## See it work

Log calls to a greeter, keep its name, then build a repeat decorator that takes a count.


In [1]:
import functools


def logged(func):
    # 1. Preserve identity for traces and help().
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # 2. Run side work, then the real call.
        print("call", func.__name__, args)
        return func(*args, **kwargs)

    return wrapper


@logged
def greet(name):
    return f"hi {name}"


print(greet("Ada"))
print(greet.__name__)


def repeat(times):
    # 3. Outer call takes decorator arguments.
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # 4. Inner wrapper uses those arguments.
            return [func(*args, **kwargs) for _ in range(times)]

        return wrapper

    return decorator


@repeat(3)
def stamp(label):
    return label


print(stamp("x"))
print(stamp.__name__)


call greet ('Ada',)
hi Ada
greet
['x', 'x', 'x']
stamp


## Check yourself

Answer these from the example above before you run the cell.


1. In the worked example, what is `greet` bound to after `@logged` runs: the original function object, or the wrapper?
2. Why does `@repeat(3)` need an outer function before the usual `decorator(func)` layer?
3. What would `help(greet)` show for the name if `wraps` were removed?

<details><summary>Answers</summary>

1. The wrapper. The original lives in the closure cell named `func`.
2. Because `@repeat(3)` must call `repeat(3)` first; that call returns the decorator that accepts `func`.
3. The name `wrapper` (and a poorer docstring), not `greet`.
</details>


In [2]:
def add_one(func):
    def wrapper(n):
        return func(n) + 1
    return wrapper


@add_one
def double(n):
    return n * 2


print(double(3))
print(double.__name__)


def add_one_wrapped(func):
    import functools

    @functools.wraps(func)
    def wrapper(n):
        return func(n) + 1
    return wrapper


@add_one_wrapped
def triple(n):
    return n * 3


print(triple.__name__)


7
wrapper
triple


<details><summary>Check your answers</summary>

`double(3)` is `7` (`3 * 2 + 1`). Without `wraps`, `__name__` is `"wrapper"`. With `wraps`, `__name__` stays `"triple"`.
</details>


## Why it works this way

Functions are ordinary objects. Passing one into another and returning a new function is the whole mechanism. `@deco` desugars to assignment after `def`, so the name in the current scope ends up bound to whatever `deco` returned. That returned object is usually a nested function that closes over the original `func`. Closure cells keep `func` alive after the decorator call finishes. Deep dive: [Scopes and closures](../../internals/06-scopes/01-closures-lesson.ipynb).

Without `functools.wraps`, the wrapper's `__name__`, `__doc__`, and signature metadata stay as the inner `wrapper` definition. Debugging, `help`, and some frameworks then lie about which function runs. `wraps(func)` copies those attributes onto the wrapper. The call cost of a thin decorator is one extra Python function call per invocation, plus whatever work you add. That is cheap for I/O logging and expensive if you decorate a tiny hot loop without need.

A decorator that takes arguments is three layers. `@repeat(3)` runs `repeat(3)` first and must return a decorator. That decorator receives `func` and returns `wrapper`. Trace of `@repeat(3)` on `stamp`:

| step | what runs | result |
|---|---|---|
| 1 | `repeat(3)` | `decorator` closure with `times=3` |
| 2 | `decorator(stamp)` | `wrapper` that calls `stamp` three times |
| 3 | name `stamp` rebound | callers get `wrapper` |

The design reason for `@` is readability at the definition site: the enhancement is visible next to `def`, not after a later assignment. Manual `f = deco(f)` is the same machine and is useful when you decorate selectively.

Evidence: `greet.__name__` stays `"greet"` with wraps, a logged call prints then returns, and `stamp("x")` yields a three-item list from the argument-taking decorator. Equal hashes are not the topic here; identity of the callable is.


Another way to see the cost: each wrapper frame appears in a traceback unless you use wraps-aware tooling, so missing `wraps` also hurts incident response, not only `.__name__` checks. For argument-taking decorators, forgetting to return `decorator` from the outer function makes `@repeat(3)` assign `None` or a surprising object to the name. Keep the three `return`s in mind: outer returns decorator, middle returns wrapper, wrapper returns the real result.

When you need the undecorated function, keep a reference before decoration or read `__wrapped__` after `wraps`, which stores the original. That escape hatch is why libraries can still test the raw function. Prefer small, documented decorators over deep stacks that nobody can unfold in their head.


In [3]:


def logged(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print("call", func.__name__)
        return func(*args, **kwargs)
    return wrapper


@logged
def greet(name):
    return f"hi {name}"


print(greet("Ada"))
print(greet.__name__)


def repeat(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            return [func(*args, **kwargs) for _ in range(times)]
        return wrapper
    return decorator


@repeat(2)
def stamp(label):
    return label


print(stamp("x"))
# Equivalent sugar-free form:
def stamp2(label):
    return label
stamp2 = repeat(2)(stamp2)
print(stamp2("y"))


call greet
hi Ada
greet
['x', 'x']
['y', 'y']


## When to use it

Use a decorator when many functions share the same before/after policy: logging, access checks, timing, caching. Use a plain helper call when only one site needs the behaviour. Use `functools.cache` (already met) when the policy is memoization. Don't stack opaque decorators that hide control flow you need to debug. Don't skip `wraps` on wrappers you expose.


Factory-style decorators that need configuration belong in a shared utilities module with tests for the wrapper contract: name preserved, return value forwarded, exceptions propagated. If a decorator swallows exceptions, say so in its docstring; silent catches are a common footgun.


## Common mistake

Writing a decorator and forgetting to return the wrapper, or returning the original function unchanged. Callers then see either `None` or no enhancement.


In [4]:
def broken(func):
    def wrapper(n):
        return func(n) + 1
    # forgot: return wrapper


@broken
def double(n):
    return n * 2


print(double)


None


The traceback's last line is the call on `NoneType`. Return the wrapper (and use `wraps`) so the name stays a callable.


In [5]:


def add_one(func):
    @functools.wraps(func)
    def wrapper(n):
        return func(n) + 1
    return wrapper


@add_one
def double(n):
    return n * 2


print(double(3))
print(double.__name__)


7
double


## Try it

Change `logged` so it also prints the return value after the call. Which lines change, and does `wraps` still matter for `__name__`?


## Recap

- `@deco` means `name = deco(name)` after the `def`.
- Wrappers close over the original function; use `functools.wraps`.
- Decorators with arguments are three layers: args → decorator → wrapper.

**Next:** [practice](01-decorators-practice.ipynb)
